# Notebook 04 — Evaluation

Four-way benchmark on the held-out test set:

| System | Type | Temporal structure |
|---|---|---|
| **SGP4 stale (3-day TLE)** | Physics baseline | None (analytic) |
| **MLP** | ML ablation | None (flattened window) |
| **LSTM** | ML | Recurrent |
| **Transformer** | ML | Self-attention |

**Key questions answered:**
1. Do any ML models outperform SGP4 overall?
2. Is temporal structure (LSTM/Transformer) better than a flat MLP?
3. Does the ML advantage grow as TLE age increases (the staleness hypothesis)?

**Input:** `dataset.zip` (from Notebook 02) + three `.pt` weight files (from Notebook 03)

## 0 — Environment Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running on Google Colab')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'torch', 'plotly'], check=True)
else:
    print('Running locally')

import io, json, math, pickle, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import torch
import torch.nn as nn

pio.renderers.default = 'colab' if IN_COLAB else 'notebook_connected'
DATA_DIR = Path('/content') if IN_COLAB else Path('../data/collected')
RE_KM    = 6371.0   # km, for error computation
print('All imports OK')

## 1 — Load Test Data & Model Weights

**Colab:** upload `dataset.zip` (from Notebook 02) and the three `.pt` files (`mlp_orbit.pt`, `lstm_orbit.pt`, `transformer_orbit.pt`) from Notebook 03.  
**Local:** all files loaded automatically from `data/collected/`.

In [ ]:
if IN_COLAB:
    from google.colab import files as _colab_files
    needed = ['X_test.npy', 'y_test.npy', 'y_sgp4_stale_test.npy',
              'scaler_X.pkl', 'dataset_meta.json',
              'mlp_orbit.pt', 'lstm_orbit.pt', 'transformer_orbit.pt']
    missing = [f for f in needed if not (DATA_DIR / f).exists()]
    if missing:
        print(f'Missing: {missing}')
        print('Upload dataset.zip first, then the three .pt weight files.')
        _uploaded = _colab_files.upload()
        for name, data in _uploaded.items():
            if name.endswith('.zip'):
                with zipfile.ZipFile(io.BytesIO(data)) as zf:
                    zf.extractall(DATA_DIR)
                print(f'  Extracted {name}')
            else:
                (DATA_DIR / name).write_bytes(data)
                print(f'  Saved {name}')

# Test arrays
X_test        = np.load(DATA_DIR / 'X_test.npy')
y_test        = np.load(DATA_DIR / 'y_test.npy')           # (N, H, 3)  ground truth
y_sgp4_stale  = np.load(DATA_DIR / 'y_sgp4_stale_test.npy')  # (N, H, 3)  stale SGP4 prediction

with open(DATA_DIR / 'scaler_X.pkl', 'rb') as f:
    scaler_X = pickle.load(f)
with open(DATA_DIR / 'dataset_meta.json') as f:
    meta = json.load(f)

HORIZONS     = meta['horizons']
FEATURE_COLS = meta['feature_cols']
TARGET_COLS  = meta['target_cols']
N_FEATURES   = X_test.shape[-1]
INPUT_LEN    = X_test.shape[1]
N_HORIZONS   = y_test.shape[1]
N_OUT        = N_HORIZONS * y_test.shape[2]  # H * 3

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Count windows where stale baseline is available
n_valid_stale = (~np.isnan(y_sgp4_stale[:, 0, 0])).sum()
print(f'Test set: {X_test.shape}  Device: {DEVICE}')
print(f'Horizons: {HORIZONS} min  |  Windows with stale baseline: {n_valid_stale:,}/{len(X_test):,}')

In [ ]:
# Model class definitions (duplicated from Notebook 03 so this notebook is self-contained)

class MLPOrbit(nn.Module):
    def __init__(self, input_len, n_features, hidden, n_out, dropout):
        super().__init__()
        in_dim = input_len * n_features
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden * 2), nn.LayerNorm(hidden * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden * 2, hidden), nn.LayerNorm(hidden),     nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, n_out),
        )
    def forward(self, x):
        return self.net(x.flatten(start_dim=1))

class LSTMOrbit(nn.Module):
    def __init__(self, n_features, hidden, lstm_layers, n_out, dropout):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, lstm_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if lstm_layers > 1 else 0)
        fc_in = hidden * 2
        self.head = nn.Sequential(
            nn.LayerNorm(fc_in), nn.Linear(fc_in, fc_in // 2),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(fc_in // 2, n_out))
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.head(torch.cat([h[-2], h[-1]], dim=-1))

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:d_model // 2])
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class TransformerOrbit(nn.Module):
    def __init__(self, n_features, d_model, nhead, num_encoder_layers, ff_dim, n_out, dropout):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_enc    = PositionalEncoding(d_model, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, ff_dim, dropout=dropout,
                                               batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_encoder_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model // 2),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model // 2, n_out))
    def forward(self, x):
        return self.head(self.encoder(self.pos_enc(self.input_proj(x))).mean(dim=1))

def load_checkpoint(path, device):
    return torch.load(path, map_location=device, weights_only=False)

def load_mlp(path, device):
    ckpt = load_checkpoint(path, device)
    hp = ckpt['hparams']
    m = MLPOrbit(hp['input_len'], hp['n_features'], hp['hidden'], hp['n_out'], hp['dropout'])
    m.load_state_dict(ckpt['model_state'])
    return m.to(device).eval()

def load_lstm(path, device):
    ckpt = load_checkpoint(path, device)
    hp = ckpt['hparams']
    m = LSTMOrbit(hp['n_features'], hp['hidden'], hp['lstm_layers'], hp['n_out'], hp['dropout'])
    m.load_state_dict(ckpt['model_state'])
    return m.to(device).eval()

def load_transformer(path, device):
    ckpt = load_checkpoint(path, device)
    hp = ckpt['hparams']
    m = TransformerOrbit(hp['n_features'], hp['d_model'], hp['nhead'],
                         hp['num_encoder_layers'], hp['ff_dim'], hp['n_out'], hp['dropout'])
    m.load_state_dict(ckpt['model_state'])
    return m.to(device).eval()

mlp_model  = load_mlp(         DATA_DIR / 'mlp_orbit.pt',         DEVICE)
lstm_model = load_lstm(        DATA_DIR / 'lstm_orbit.pt',        DEVICE)
tf_model   = load_transformer( DATA_DIR / 'transformer_orbit.pt', DEVICE)
print('All three models loaded.')

## 2 — Run Inference & Build Results Table

Geodetic error is computed as the great-circle distance on the surface (lat/lon) plus altitude difference added in quadrature — giving a combined 3-D position error in km.

In [ ]:
def geodetic_error_km(pred, true):
    """
    3-D position error in km.
    pred, true: (..., 3) arrays with columns [lat_deg, lon_deg, alt_km].
    Uses great-circle surface distance + altitude difference in quadrature.
    """
    lat1 = np.radians(pred[..., 0]); lat2 = np.radians(true[..., 0])
    lon1 = np.radians(pred[..., 1]); lon2 = np.radians(true[..., 1])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    d_surface = 2 * (RE_KM + true[..., 2]) * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    d_alt = pred[..., 2] - true[..., 2]
    return np.sqrt(d_surface**2 + d_alt**2)

def predict_all(model, X, batch=512):
    """Batch inference. Returns (N, H, 3)."""
    parts = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            Xb = torch.tensor(X[i:i+batch], dtype=torch.float32).to(DEVICE)
            parts.append(model(Xb).cpu().numpy())
    return np.concatenate(parts, axis=0).reshape(-1, N_HORIZONS, 3)

# --- Recover unscaled last timestep for TLE-age analysis ---
X_test_unscaled = scaler_X.inverse_transform(
    X_test.reshape(-1, N_FEATURES)).reshape(X_test.shape)
tle_age_col  = FEATURE_COLS.index('tle_age_days')
tle_age_last = X_test_unscaled[:, -1, tle_age_col]   # TLE age at end of each window

# --- Run inference ---
mlp_pred  = predict_all(mlp_model,  X_test)
lstm_pred = predict_all(lstm_model, X_test)
tf_pred   = predict_all(tf_model,   X_test)

# --- Mask windows where stale baseline is NaN ---
stale_mask = ~np.isnan(y_sgp4_stale[:, 0, 0])   # True where stale data is available

print(f'MLP  pred shape : {mlp_pred.shape}')
print(f'LSTM pred shape : {lstm_pred.shape}')
print(f'TF   pred shape : {tf_pred.shape}')
print(f'Stale mask      : {stale_mask.sum():,} valid rows (of {len(stale_mask):,})')

## 3 — Metric Table (MAE / RMSE by horizon)

In [ ]:
rows = []
for hi, h in enumerate(HORIZONS):
    for model_name, preds, mask in [
        ('SGP4 stale (3-day)', y_sgp4_stale,  stale_mask),   # only valid stale rows
        ('MLP',                mlp_pred,       stale_mask),   # same rows for fair comparison
        ('LSTM',               lstm_pred,      stale_mask),
        ('Transformer',        tf_pred,        stale_mask),
    ]:
        err = geodetic_error_km(preds[mask, hi, :], y_test[mask, hi, :])
        rows.append({'Horizon (min)': h, 'Model': model_name,
                     'MAE (km)': round(float(err.mean()), 3),
                     'RMSE (km)': round(float(np.sqrt((err**2).mean())), 3),
                     'n': int(mask.sum())})

metrics_df = pd.DataFrame(rows)

# Pivot for easy reading
pivot_mae  = metrics_df.pivot_table(index='Model', columns='Horizon (min)', values='MAE (km)',  aggfunc='first')
pivot_rmse = metrics_df.pivot_table(index='Model', columns='Horizon (min)', values='RMSE (km)', aggfunc='first')

print('=== MAE (km) ===')
print(pivot_mae.to_string())
print('\n=== RMSE (km) ===')
print(pivot_rmse.to_string())

# Bar chart
fig = px.bar(metrics_df, x='Horizon (min)', y='MAE (km)', color='Model', barmode='group',
             title='MAE (km) by Forecast Horizon — All Systems',
             template='plotly_white',
             color_discrete_map={'SGP4 stale (3-day)': 'grey', 'MLP': 'orange',
                                 'LSTM': 'royalblue', 'Transformer': 'firebrick'})
fig.show()

## 4 — Error vs TLE Age

Does the ML model outperform SGP4 when the TLE is stale?  
Split test samples by TLE age quartile and compare MAE at T+60 min.

In [ ]:
H60_IDX   = HORIZONS.index(60)
age_bins   = [0, 1, 3, 7, 100]
age_labels = ['<1 day', '1–3 days', '3–7 days', '>7 days']
age_group  = pd.cut(tle_age_last[stale_mask], bins=age_bins, labels=age_labels)

age_rows = []
for grp in age_labels:
    local_mask = (age_group == grp)
    if local_mask.sum() == 0:
        continue
    for model_name, preds in [
        ('SGP4 stale (3-day)', y_sgp4_stale[stale_mask]),
        ('MLP',                mlp_pred[stale_mask]),
        ('LSTM',               lstm_pred[stale_mask]),
        ('Transformer',        tf_pred[stale_mask]),
    ]:
        err = geodetic_error_km(preds[local_mask, H60_IDX, :], y_test[stale_mask][local_mask, H60_IDX, :])
        age_rows.append({'TLE Age': grp, 'Model': model_name,
                         'MAE (km)': float(err.mean()), 'n': int(local_mask.sum())})

age_df = pd.DataFrame(age_rows)
fig = px.bar(age_df, x='TLE Age', y='MAE (km)', color='Model', barmode='group',
             title='MAE at T+60 min stratified by TLE Age — Staleness Hypothesis Test',
             labels={'MAE (km)': 'MAE (km)', 'TLE Age': 'TLE Age at Window End'},
             category_orders={'TLE Age': age_labels},
             template='plotly_white',
             color_discrete_map={'SGP4 stale (3-day)': 'grey', 'MLP': 'orange',
                                 'LSTM': 'royalblue', 'Transformer': 'firebrick'})
fig.show()

pivot_age = age_df.pivot_table(index='TLE Age', columns='Model', values='MAE (km)').round(3)
print(pivot_age.reindex(age_labels).to_string())

## 5 — Example Prediction Trajectories

Visualise 3 random test windows: ground truth vs SGP4 stale vs MLP vs LSTM vs Transformer on a lat/lon map. Windows are sampled from rows where the stale baseline is available.

In [ ]:
N_EXAMPLES = 3
# Pick random windows where the stale baseline is available
rng = np.random.default_rng(42)
valid_indices = np.where(stale_mask)[0]
example_idxs  = rng.choice(valid_indices, size=min(N_EXAMPLES, len(valid_indices)), replace=False)

for i, idx in enumerate(example_idxs):
    fig = go.Figure()

    # Ground truth
    gt = y_test[idx]          # (H, 3)
    fig.add_trace(go.Scatter(x=gt[:, 1], y=gt[:, 0], mode='lines+markers',
                             name='Ground truth', line=dict(color='black', width=2)))

    # SGP4 stale
    sg = y_sgp4_stale[idx]    # (H, 3)
    fig.add_trace(go.Scatter(x=sg[:, 1], y=sg[:, 0], mode='lines+markers',
                             name='SGP4 stale', line=dict(color='grey', dash='dot')))

    # MLP
    mp = mlp_pred[idx]
    fig.add_trace(go.Scatter(x=mp[:, 1], y=mp[:, 0], mode='lines+markers',
                             name='MLP', line=dict(color='orange', dash='dash')))

    # LSTM
    lp = lstm_pred[idx]
    fig.add_trace(go.Scatter(x=lp[:, 1], y=lp[:, 0], mode='lines+markers',
                             name='LSTM', line=dict(color='royalblue', dash='dash')))

    # Transformer
    tp = tf_pred[idx]
    fig.add_trace(go.Scatter(x=tp[:, 1], y=tp[:, 0], mode='lines+markers',
                             name='Transformer', line=dict(color='firebrick', dash='dash')))

    # Context — last 10 input timesteps for orientation
    ctx = X_test_unscaled[idx, -10:]
    lat_col = FEATURE_COLS.index('lat')
    lon_col = FEATURE_COLS.index('lon')
    fig.add_trace(go.Scatter(x=ctx[:, lon_col], y=ctx[:, lat_col], mode='lines',
                             name='Input context', line=dict(color='green', width=1, dash='dot'),
                             opacity=0.5))

    age_str = f'{tle_age_last[idx]:.1f}d'
    fig.update_layout(
        title=f'Example {i+1} — window #{idx}  (TLE age ≈ {age_str})',
        xaxis_title='Longitude (°)', yaxis_title='Latitude (°)',
        template='plotly_white', height=450)
    fig.show()